# Analyse des commentaires
Dans ce notebook, nous allons regarder en détail les commentaires laissés par les utilisateurs.
Le travail sera divisé en deux parties : Construction du corpus et Début ? d'analyse des fréquences

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from boardgames_recsys.data.filtering import filter_df
import boardgames_recsys.text.filtering as ft
from boardgames_recsys.data.matrix import center_score
import seaborn as sns

%load_ext autoreload
%autoreload 2

## Partie 1 : Construction corpus

Afin de construire le corpus, nous allons utiliser la fonction ```corpus_construction``` du fichier _text_filtering.py_ Celle-ci prend en paramètre la taille du corpus souhaité et utilise la BDD _lemmas.csv_ afin de récupérer les fréquences d'apparition de chaque lemma. Pour l'instant, nous gardons les $k$ instances les plus fréquentes, où $k$ représente la taille passée en paramètre.

(proposition de modification: ajouter en paramètre le nombre d'apparition minimum de chaque instance sur l'ensemble des avis ?)

In [ ]:
corpus2 = ft.construction_corpus(2000)
# corpus5 = ft.construction_corpus(5000)
# corpus10 = ft.construction_corpus(10000)

In [ ]:
lemmas = pd.read_csv("generated_data/lemmas.csv", index_col=0)

## Partie 2 : Analyse des fréquences

Maintenant que nous avons créé nos corpus, nous pouvons comparer les champs lexicaux utilisés sur les avis négatifs et positifs. Pour cela, nous devons d'abord séparer les avis en deux catégories. Il est important de centrer les notes afin d'enlever le biais des utilisateurs sur les notes.

In [ ]:
# Construction de la BDD avec avis centrés
avis = pd.read_csv("database_cleaned/avis_clean.csv", index_col = 0)
min_reviews = 10
rev_filter = filter_df(avis, min_reviews)
filtered_centrd_data, _= center_score(rev_filter)

In [ ]:
folder = "database_cleaned"
jeux_clean  = pd.read_csv(f"{folder}/jeux_clean.csv", index_col=0)

In [ ]:
# Séparation de la bdd 
positifs = filtered_centrd_data[filtered_centrd_data['Rating'] >=0][['Comment title', 'Comment body']].copy()
negatifs = filtered_centrd_data[filtered_centrd_data['Rating'] <0][['Comment title', 'Comment body']].copy()

In [ ]:
print("Nombre d'avis negatif", len(negatifs)/len(filtered_centrd_data))
print("Nombre d'avis positif", len(positifs)/len(filtered_centrd_data))

In [ ]:
# lem = pd.read_csv("generated_data/lemmas.csv")[['Comment line', 'Lemma']]
# lem['Lemma'] = lem['Lemma'].apply(lambda val : " " if type(val) != str else val )
# lem = lem.groupby(by='Comment line').apply(lambda row : " ".join(row["Lemma"])) 
# lem = lem.to_frame()
# lem.columns = ['Comment']
# lem.to_csv("generated_data/avis_lemmatized.csv")

À partir de là, nous pouvons appliquer la fonction ```word_freq``` qui calcul la fréquence de chaque lemma dans chaque dataframe des avis.

In [ ]:
fpos = ft.words_freq(positifs, corpus2)
fneg = ft.words_freq(negatifs, corpus2)

In [ ]:
fpos = fpos.sort_values(by=['Freq'], ascending = False)
fneg = fneg.sort_values(by=['Freq'], ascending = False)

In [ ]:
sns.set_theme(rc={'figure.figsize' : (12, 5)})
ax = sns.lineplot(fpos.head(90),x='Lemma',y='Freq',label='avis positifs')
ax = sns.lineplot(fneg.head(90),x='Lemma',y='Freq',label='avis négatifs')
ax.tick_params(axis='x', rotation=90, labelsize=8)
ax.set_title("Fréquence des Lemmas par type d'avis : corpus 2000")

On peut remarquer qu'à certains endroit, il existe des écarts de fréquence beaucoup plus prononcés qu'à d'autres. Afin de mieux les voir, nous pouvons utiliser la fonction ```diff_freq```. Elle retourne un dataframe avec, pour chaque lemma $l$, la différence entre la fréquence de $l$ dans les avis positifs et celle dans les avis négatifs. Ainsi, la différence de fréquence sera positive si $l$ apparaît plus souvent dans les avis positifs et que négatifs, et vice versa.

Le DataFrame retourné est trié par différence de fréquence décroissante.

In [ ]:
fdiff = ft.diff_freq(fpos,fneg) # si valeur >= 0, alors + grande frequence dans fpos que fneg 
fdiff

On remarque que pour 2000 mots, tous les mots dans fpos sont aussi dans fneg

In [ ]:
sns.set_theme(rc={'figure.figsize' : (12, 5)})
ax = sns.lineplot(fdiff.head(90), x='Lemma',y='Freq differency', label='1')
ax.tick_params(axis='x', rotation=90, labelsize=8)
ax.set_title("Différence de fréquences : 50 mots + notables avis positifs : corpus 2000")

In [ ]:
sns.set_theme(rc={'figure.figsize' : (12, 5)})
ax = sns.lineplot(fdiff.tail(90)[::-1], x='Lemma',y='Freq differency', label='1')
ax.tick_params(axis='x', rotation=90, labelsize=8)
ax.set_title("Différence de fréquence : 50 mots + notables avis négatifs : corpus 2000")

Zoom sur les différences de fréquence


In [ ]:
lemmas_neg = fdiff.tail(90)['Lemma'].to_numpy()
pos_tail = fpos[fpos['Lemma'].isin(lemmas_neg)]
neg_tail = fneg[fneg['Lemma'].isin(lemmas_neg)]

ax = sns.lineplot(pos_tail, x='Lemma', y='Freq', label='avis positifs')
ax = sns.lineplot(neg_tail, x='Lemma', y='Freq', label='avis négatifs')
ax.tick_params(axis='x', rotation=90, labelsize=8)
ax.set_title("Différence de fréquence sur les pires mots : corpus 2000")

In [ ]:
lemmas_neg = fdiff.head(90)['Lemma'].to_numpy()
pos_head = fpos[fpos['Lemma'].isin(lemmas_neg)]
neg_head = fneg[fneg['Lemma'].isin(lemmas_neg)]

ax = sns.lineplot(pos_head, x='Lemma', y='Freq', label='avis positifs')
ax = sns.lineplot(neg_head, x='Lemma', y='Freq', label='avis négatifs')
ax.tick_params(axis='x', rotation=90, labelsize=8)
ax.set_title("Différence de fréquence sur les meilleurs mots : corpus 2000")

Visualisation par types de mots

In [ ]:
lemmas_sp = lemmas[~lemmas["Lemma"].isna()]
lemmas_sp = lemmas_sp[lemmas_sp['Part of speech'].isin(['ADJ', 'NOM', "VER:infi","VER:pper", "VER:pres"])]
lemmas_sp_np = lemmas_sp[['Lemma', "Part of speech"]].to_numpy()

In [ ]:
lemma_speech = lemmas_sp[['Lemma', 'Part of speech']]
lemma_speech['Part of speech'].unique()

In [ ]:
# get only verbs
lemma_verb = lemma_speech[lemma_speech['Part of speech'].isin(['VER:infi', 'VER:pper', 'VER:pres'])]
fdiff_verb = fdiff[fdiff['Lemma'].isin(lemma_verb['Lemma'].unique())].sort_values(by=['Freq differency'], ascending=False)
lemma_verb['Lemma'].unique()

In [ ]:
fdiff_verb.shape

In [ ]:
sns.set_theme(rc={'figure.figsize' : (12, 5)})
ax = sns.lineplot(fdiff_verb.head(90), x='Lemma',y='Freq differency', label='word frequence')
plt.title("Verbs most frequent in positive reviews")
ax.tick_params(axis='x', rotation=90, labelsize=8)

In [ ]:
sns.set_theme(rc={'figure.figsize' : (12, 5)})
ax = sns.lineplot(fdiff_verb.tail(90)[::-1], x='Lemma',y='Freq differency', label='word frequence')
plt.title("Verbs most frequent in negative reviews")
ax.tick_params(axis='x', rotation=90, labelsize=8)

In [ ]:
# get only adj
lemma_adj = lemma_speech[lemma_speech['Part of speech'].isin(['ADJ'])]
fdiff_adj = fdiff[fdiff['Lemma'].isin(lemma_adj['Lemma'].unique())].sort_values(by=['Freq differency'], ascending=False)
lemma_adj['Lemma'].unique()

In [ ]:
sns.set_theme(rc={'figure.figsize' : (12, 5)})
ax = sns.lineplot(fdiff_adj.head(90), x='Lemma',y='Freq differency', label='word frequence')
plt.title("Adj most frequent in positive reviews")
ax.tick_params(axis='x', rotation=90, labelsize=8)

In [ ]:
sns.set_theme(rc={'figure.figsize' : (12, 5)})
ax = sns.lineplot(fdiff_adj.tail(90)[::-1], x='Lemma',y='Freq differency', label='word frequence')
plt.title("Adj most frequent in negative reviews")
ax.tick_params(axis='x', rotation=90, labelsize=8)

# color one label
for label, position in zip(ax.get_xticklabels(), ax.get_xticks()):
    if label.get_text() == 'enfantin':
        label.set_color("blue")
        label.set_fontweight('bold')

In [ ]:
# get only nom
lemma_nom = lemma_speech[lemma_speech['Part of speech'].isin(['NOM'])]
fdiff_nom = fdiff[fdiff['Lemma'].isin(lemma_nom['Lemma'].unique())].sort_values(by=['Freq differency'], ascending=False)
lemma_nom['Lemma'].unique()

In [ ]:
sns.set_theme(rc={'figure.figsize' : (12, 5)})
ax = sns.lineplot(fdiff_nom.head(90), x='Lemma',y='Freq differency', label='word frequence')
plt.title("Nom most frequent in positive reviews")
ax.tick_params(axis='x', rotation=90, labelsize=8)

In [ ]:
sns.set_theme(rc={'figure.figsize' : (12, 5)})
ax = sns.lineplot(fdiff_nom.tail(90)[::-1], x='Lemma',y='Freq differency', label='word frequence')
plt.title("Nom most frequent in negative reviews")
ax.tick_params(axis='x', rotation=90, labelsize=8)

In [ ]:
# vocabulary + - difference
fdiff[fdiff["Freq differency"] >0].shape, fdiff[fdiff["Freq differency"] <=0].shape

### See most and less rated games vocabulary 

In [ ]:
# sort games by ratings rank
rank_game = rev_filter[["Game id", "Rating"]].groupby("Game id").mean().sort_values(by="Rating", ascending=False)
rank_game.head(), rank_game.tail()

In [ ]:
# rev_filter[rev_filter["Game id"] == 1928]

In [ ]:
top_5 = rev_filter[rev_filter["Game id"].isin(rank_game.head(6).index)][["Game id", "Comment body"]]
top_5['Comment body'] = top_5['Comment body'].apply(lambda row : row.split())
top_5 = top_5.explode(column='Comment body')
top_5 = top_5[top_5["Comment body"].isin(corpus2)]
top_5

In [ ]:
# word frequencies for these games
def word_frequencies(group):
    words = group['Comment body'].values
    val, nb = np.unique(words, return_counts=True)
    return pd.Series(val, index=nb/np.sum(nb))

result = top_5.groupby('Game id').apply(word_frequencies).reset_index(name="word").rename(columns={'level_1': 'freq'})
result

In [ ]:
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(12, 8))

for i, game_id in enumerate(top_5['Game id'].unique()):
    game_words = result[result['Game id'] == game_id][["word", "freq"]]
    
    ax = axes[i // 3, i % 3]
    sns.barplot(x='freq', y='word', data=game_words.nlargest(20, columns="freq"), ax=ax)

    ax.set_title(f"Game {game_id}")
    ax.set_ylabel('Word')
    ax.set_xlabel('Frequency')

fig.suptitle('Top rated games Word frequency')
plt.tight_layout()
plt.show()

In [ ]:
for id in result["Game id"].unique():
    print(id, jeux_clean.iloc[id]["Type"]," : ", jeux_clean.iloc[id]["Game name website"])

We can categorize the games that have NaN categories, 

-> rallyman-dirt is about cars and race

-> dominion-age-des-tenebres-0 games with cards, kingdoms, battle, is an extension

In [ ]:
res_clean = result.drop(result[result["word"].isin(['un', 'jeu', 'jouer', 'joue', 'faire'])].index)
res_clean.shape, result.shape

In [ ]:
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(12, 8))

for i, game_id in enumerate(result['Game id'].unique()):
    game_words = res_clean[res_clean['Game id'] == game_id][["word", "freq"]]
    
    ax = axes[i // 3, i % 3]
    sns.barplot(x='freq', y='word', data=game_words.nlargest(20, columns="freq"), ax=ax)

    ax.set_title(f"Game {game_id}")
    ax.set_ylabel('Word')
    ax.set_xlabel('Frequency')

fig.suptitle('Top rated games Word frequency')
plt.tight_layout()
plt.show()

In [ ]:
word_pos = fdiff[fdiff['Freq differency'] >= 0]
word_neg = fdiff[fdiff['Freq differency'] < 0]
word_pos.shape, word_neg.shape

In [ ]:
# percentage of words that are in + vocabulary
result.groupby("Game id")["word"].apply(lambda x : x.isin(word_pos['Lemma']).sum()/len(x))

In [ ]:
# check by category of words
print("Percentage of + words among NOM")
print(result[result['word'].isin(lemma_nom['Lemma'])].groupby("Game id")["word"].apply(lambda x : x.isin(word_pos['Lemma']).sum()/len(x)))
print("\nPercentage of + words among ADJ")
print(result[result['word'].isin(lemma_adj['Lemma'])].groupby("Game id")["word"].apply(lambda x : x.isin(word_pos['Lemma']).sum()/len(x)))
print("\nPercentage of + words among VERB")
print(result[result['word'].isin(lemma_verb['Lemma'])].groupby("Game id")["word"].apply(lambda x : x.isin(word_pos['Lemma']).sum()/len(x)))

In [ ]:
# worst games
worst_5 = rev_filter[rev_filter["Game id"].isin(rank_game.tail(6).index)][["Game id", "Comment body"]]
worst_5['Comment body'] = worst_5['Comment body'].apply(lambda row : row.split())
worst_5 = worst_5.explode(column='Comment body')
worst_5 = worst_5[worst_5["Comment body"].isin(corpus2)]
worst_5

In [ ]:
result = worst_5.groupby('Game id').apply(word_frequencies).reset_index(name="word").rename(columns={'level_1': 'freq'})
result

In [ ]:
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(12, 8))

for i, game_id in enumerate(worst_5['Game id'].unique()):
    game_words = result[result['Game id'] == game_id][["word", "freq"]]
    
    ax = axes[i // 3, i % 3]
    sns.barplot(x='freq', y='word', data=game_words.nlargest(20, columns="freq"), ax=ax)

    ax.set_title(f"Game {game_id}")
    ax.set_ylabel('Word')
    ax.set_xlabel('Frequency')

fig.suptitle('Worst rated games Word frequency')
plt.tight_layout()
plt.show()

In [ ]:
for id in result["Game id"].unique():
    print(id, jeux_clean.iloc[id]["Type"]," : ", jeux_clean.iloc[id]["Game name website"])

In [ ]:
res_clean = result.drop(result[result["word"].isin(['un', 'jeu', 'jouer', 'joue', 'faire'])].index)
res_clean.shape, result.shape

In [ ]:
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(12, 8))

for i, game_id in enumerate(worst_5['Game id'].unique()):
    game_words = res_clean[res_clean['Game id'] == game_id][["word", "freq"]]
    
    ax = axes[i // 3, i % 3]
    sns.barplot(x='freq', y='word', data=game_words.nlargest(20, columns="freq"), ax=ax)

    ax.set_title(f"Game {game_id}")
    ax.set_ylabel('Word')
    ax.set_xlabel('Frequency')

fig.suptitle('Worst rated games Word frequency')
plt.tight_layout()
plt.show()

In [ ]:
# percentage of words that are in + vocabulary
result.groupby("Game id")["word"].apply(lambda x : x.isin(word_pos['Lemma']).sum()/len(x))

In [ ]:
# check by category of words
print("Percentage of + words among NOM")
print(result[result['word'].isin(lemma_nom['Lemma'])].groupby("Game id")["word"].apply(lambda x : x.isin(word_pos['Lemma']).sum()/len(x)))
print("\nPercentage of + words among ADJ")
print(result[result['word'].isin(lemma_adj['Lemma'])].groupby("Game id")["word"].apply(lambda x : x.isin(word_pos['Lemma']).sum()/len(x)))
print("\nPercentage of + words among VERB")
print(result[result['word'].isin(lemma_verb['Lemma'])].groupby("Game id")["word"].apply(lambda x : x.isin(word_pos['Lemma']).sum()/len(x)))

In [ ]:
# take mean of a game and compare the percentage of pos (>mean) and neg (<mean) reviews -> not very significant
def perc_mean_rev(df, corpus, pos_corp, neg_corp, verbose=False):
    """df (not centered) contains columns 'Comment body' for 1 game 
    return percentage of positive words in positive review (for neg, 1 - value)
    and percentage of negative words in negative review (for pos, 1 - value) 
    """
    df.loc[:,"Comment body"] = df["Comment body"].apply(lambda x : x.split())
    if verbose : print("Mean Rating", df['Rating'].mean())
    df.loc[:,'Rating'] -= df['Rating'].mean()
    if verbose : print("Number of positive review", len(df[df['Rating'] >= 0])/len(df), "\nNumber of negative review", len(df[df['Rating'] < 0])/len(df))

    tokens = df["Comment body"].explode()
    tokens = tokens[tokens.isin(corpus)] # keep corpus words
    freq_pos = tokens.isin(pos_corp).sum()
    freq_neg = tokens.isin(neg_corp).sum()
    if verbose : 
        print("Number of positif word in positive comments", freq_pos/(freq_pos + freq_neg))
        print("Number of negatif word in negativ comments", freq_neg/(freq_pos + freq_neg))

    return freq_pos/(freq_pos + freq_neg), freq_neg/(freq_pos + freq_neg)

pos_adj, neg_adj = lemma_adj[lemma_adj['Lemma'].isin(word_pos['Lemma'])].drop_duplicates()['Lemma'], lemma_adj[lemma_adj['Lemma'].isin(word_neg['Lemma'])].drop_duplicates()['Lemma']
pos_verb, neg_verb = lemma_verb[lemma_verb['Lemma'].isin(word_pos['Lemma'])].drop_duplicates()['Lemma'], lemma_verb[lemma_verb['Lemma'].isin(word_neg['Lemma'])].drop_duplicates()['Lemma']
pos_nom, neg_nom = lemma_nom[lemma_nom['Lemma'].isin(word_pos['Lemma'])].drop_duplicates()['Lemma'], lemma_nom[lemma_nom['Lemma'].isin(word_neg['Lemma'])].drop_duplicates()['Lemma']

In [ ]:
perc_mean_rev(avis[avis["Game id"] == 6179][['Comment body', "Rating"]], corpus2, pos_adj, neg_adj, verbose=True)

In [ ]:
perc_mean_rev(avis[avis["Game id"] == 3600][['Comment body', "Rating"]], corpus2, pos_adj, neg_adj, verbose=True)

In [ ]:
avg_posw_rev = 0
avg_negw_rev = 0

# average on all words

for id_game in filtered_centrd_data["Game id"].unique():
    pos_w, neg_w = perc_mean_rev(avis[avis["Game id"] == id_game][['Comment body', "Rating"]], corpus2, word_pos['Lemma'], word_neg['Lemma'])
    avg_posw_rev, avg_negw_rev = avg_posw_rev + pos_w, avg_negw_rev + neg_w

avg_posw_rev/filtered_centrd_data["Game id"].unique().shape[0], avg_negw_rev/filtered_centrd_data["Game id"].unique().shape[0]

In [ ]:
# given a user and a rated game -> if rated > mean = liked, if rated < mean = disliked
# print the percentage + and - words for the 2 groups of games

def user_vocab(id, df, corpus, pos_corp, neg_corp, verbose = False):
    """ df with centered ratings, columns : "Rating", "Comment body", "Game id"
    calculate the frequence of words in neg and pos comments 

    return percentage of positive (resp. neg) words for positive (resp. neg) reviews
    """
    pos_rev, neg_rev = df[df["Rating"] >= 0]["Comment body"].apply(lambda x: x.split()), df[df["Rating"] < 0]["Comment body"].apply(lambda x: x.split())
    # if len(pos_rev) == 0 or len(neg_rev) == 0: # case all rating = 0
    #     return
    tok_pos, tok_neg = pos_rev.explode(), neg_rev.explode()
    tok_pos, tok_neg = tok_pos[tok_pos.isin(corpus)], tok_neg[tok_neg.isin(corpus)] # tokens from +- comments that are in the corpus      
    
    # we see the percentage of these words in the +- corp
    if verbose:
        print("Positive reviews")
        print("Percentage of positive and negative words", tok_pos.isin(pos_corp).sum()/len(tok_pos), tok_pos.isin(neg_corp).sum()/len(tok_pos))
        print("Negative reviews")
        print("Percentage of positive and negative words", tok_neg.isin(pos_corp).sum()/len(tok_neg), tok_neg.isin(neg_corp).sum()/len(tok_neg))

    return tok_pos.isin(pos_corp).sum()/len(tok_pos), tok_neg.isin(neg_corp).sum()/len(tok_neg)
    

In [ ]:
user_vocab(4, filtered_centrd_data[(filtered_centrd_data["User id"]== 4)], corpus2, word_pos['Lemma'], word_neg['Lemma'], verbose=True)

In [ ]:
# average on all users

avg_posw_rev_u = 0
avg_negw_rev_u = 0

# average on all words
a = filtered_centrd_data["User id"].unique()
# fcd = filtered_centrd_data[filtered_centrd_data["User id"].isin(a)]

a.sort()
for id_user in a:
    if id_user in [2514, 7351]: # get nan
        continue
    pos_w, neg_w = user_vocab(id_user,filtered_centrd_data[filtered_centrd_data["User id"] == id_user][['Comment body', "Rating"]], corpus2, word_pos['Lemma'], word_neg['Lemma'])
    avg_posw_rev_u, avg_negw_rev_u = avg_posw_rev_u + pos_w, avg_negw_rev_u + neg_w

avg_posw_rev_u/filtered_centrd_data["User id"].unique().shape[0], avg_negw_rev_u/filtered_centrd_data["User id"].unique().shape[0]

In [ ]:
avg_posw_rev_u/filtered_centrd_data["User id"].unique().shape[0], 1 - avg_negw_rev_u/filtered_centrd_data["User id"].unique().shape[0]

In [ ]:
# get the most frequent words freq per type

In [ ]:
avis_jeux = filtered_centrd_data.merge(jeux_clean, on="Game id")[["Game id", "Comment body", "Type"]].dropna() # no nans atm
avis_jeux

In [ ]:
avis_jeux

In [ ]:
# for each category create a new row
types = avis_jeux['Type'].str.split('|').explode()
user_game_type = pd.DataFrame({
    #'User id': avis_jeux['User id'].repeat(avis_jeux['Type'].str.split('|').apply(len)),
    'Game id': avis_jeux['Game id'].repeat(avis_jeux['Type'].str.split('|').apply(len)),
    'Comment body': avis_jeux['Comment body'].repeat(avis_jeux['Type'].str.split('|').apply(len)),
    'Type': types
})


In [ ]:
user_game_type["Comment body"] = user_game_type["Comment body"].apply(lambda x: x.split())
user_game_type

In [ ]:
user_game_type = user_game_type.explode(column='Comment body')
user_game_type

In [ ]:
user_game_type = user_game_type[user_game_type["Comment body"].isin(corpus2)]
user_game_type

In [ ]:
res = user_game_type.groupby("Type").apply(word_frequencies)

In [ ]:
res = res.reset_index(name="word").rename(columns={"level_1": "freq"})

In [ ]:
res

In [ ]:
res["Type"].unique()

In [ ]:
cat = ['Antiquité','Jeux de dés','Hasard (Dé, Cartes, ...)','Jeux de pions','Jeux de plateau' ,'Gestion']

In [ ]:
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(12, 8))

for i, c in enumerate(cat):
    grp = res[res["Type"] == c][["freq", "word"]]
    grp = (grp.sort_values(ascending=False, by="freq"))

    ax = axes[i // 3, i % 3]
    sns.barplot(x='freq', y='word', data=grp.nlargest(20, columns="freq"), ax=ax)

    ax.set_title(f"Category {c}")
    ax.set_ylabel('Word')
    ax.set_xlabel('Frequency')

fig.suptitle('Most frequent words per category')
plt.tight_layout()
plt.show()

In [ ]:
res_clean = res.drop(res[res["word"].isin(['un', 'jeu', 'jouer', 'joue', 'faire', 'fois', 'car'])].index)
res_clean.shape, res.shape

In [ ]:
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(12, 8))

for i, c in enumerate(cat):
    grp = res_clean[res_clean["Type"] == c][["freq", "word"]]
    grp = grp[grp["word"].isin(lemma_verb['Lemma'])]
    grp = (grp.sort_values(ascending=False, by="freq"))

    ax = axes[i // 3, i % 3]
    sns.barplot(x='freq', y='word', data=grp.nlargest(20, columns="freq"), ax=ax)

    ax.set_title(f"Category {c}")
    ax.set_ylabel('Word')
    ax.set_xlabel('Frequency')

fig.suptitle('Most frequent words per category')
plt.tight_layout()
plt.show()